<a href="https://colab.research.google.com/github/Bryce-Neuman/BG885-Assignment-7-NeumanB-Discussion/blob/main/Assignment_GENBUS_745_RPA_Neuman_Bryce_T.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This script will print the weather along with sunrise sunset for an entered zipcode. I am going to have Sun Prarie but this could be changed.

In [1]:
pip install requests pandas astral pgeocode

In [2]:
#Import request to make HTTP requests
import requests
#Import pandas for datafram structure
import pandas as pd
import datetime as dt
from zoneinfo import ZoneInfo
#Import to be able to take zip code to GPS location
import pgeocode
from astral import LocationInfo
#Use for sunrise sunset
from astral.sun import sun


In [3]:
# US based ZIP code
ZIP = "54212"  # Fish Creek, WI (Location of Peninsula State Park)

# Get lat/lon from ZIP above with all being US based zips
nomi = pgeocode.Nominatim("us")
info = nomi.query_postal_code(ZIP)
lat, lon = float(info.latitude), float(info.longitude)

#  Get NWS grid/forecast and timezone
points = requests.get(f"https://api.weather.gov/points/{lat},{lon}").json()
forecast_url = points["properties"]["forecast"]
tz = points["properties"].get("timeZone", "America/Chicago")

# Fetch forecast periods
forecast = requests.get(forecast_url).json()
periods = forecast["properties"]["periods"]

# 4) Aggregate daily high/low using period start times
highs = {}
lows = {}
# Loops to get highs and lows from data, takes the high daytime temp and adds it and takes the low nighttime temp
for p in periods:
    start = pd.to_datetime(p["startTime"])
    date = start.date()
    temp = p.get("temperature")
    if p.get("isDaytime"):
        highs[date] = max(temp, highs.get(date, -9999))
    else:
        lows[date] = min(temp, lows.get(date, 9999))

# Compute sunrise/sunset per date with Astral
location = LocationInfo(name="SunPrairie", region="US", timezone=tz, latitude=lat, longitude=lon)
rows = []
all_dates = sorted(set(list(highs.keys()) + list(lows.keys())))
#Loops through dates to build table
for d in all_dates:
    s = sun(location.observer, date=d, tzinfo=ZoneInfo(tz))
    sunrise = s["sunrise"]
    sunset = s["sunset"]
    rows.append({
        "Date": d.isoformat(),
        "Day":d.strftime("%A"),
        "High (°F)": highs.get(d),
        "Low (°F)": lows.get(d),
        "Sunrise": sunrise.strftime("%I:%M %p %Z"),
        "Sunset": sunset.strftime("%I:%M %p %Z"),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

      Date       Day  High (°F)  Low (°F)      Sunrise       Sunset
2026-08-02    Sunday        NaN      56.0 05:35 AM CDT 08:14 PM CDT
2026-08-03    Monday       79.0      62.0 05:36 AM CDT 08:12 PM CDT
2026-08-04   Tuesday       82.0      61.0 05:37 AM CDT 08:11 PM CDT
2026-08-05 Wednesday       78.0      60.0 05:38 AM CDT 08:09 PM CDT
2026-08-06  Thursday       80.0      62.0 05:40 AM CDT 08:08 PM CDT
2026-08-07    Friday       82.0      65.0 05:41 AM CDT 08:07 PM CDT
2026-08-08  Saturday       81.0      63.0 05:42 AM CDT 08:05 PM CDT
2026-08-09    Sunday       80.0       NaN 05:43 AM CDT 08:04 PM CDT
